In [27]:
# 01. 상태키 추가하기.
# 먼저 라이브러리들을 로드하고, 상태 구조를 정의함. messages 필드에 name과 release_date 필드를 추가함.

from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from dotenv import load_dotenv

load_dotenv()

llm = init_chat_model("openai:gpt-4o")

# State라는 이름의 딕셔너리는 반드시 messages, name, release_date 라는 Key를 가져야한다고 선언하는 것임.
class State(TypedDict):
    messages: Annotated[list, add_messages]
    name: str
    release_date: str
#   . State: 랭그래프에서 워크플로우 전반에 걸쳐 공유되는 상태를 정의함.
#   . messages: 챗봇의 대화 메시지를 저장하는 기본 필드임.
#   . name, release_date: 정형화된 정보를 저장하는데 사용하는 사용자 정의 필드임.

# 상태에 명시적으로 값을 저장하면 다른 노드에서도 해당 데이터를 손쉽게 참조하거나 활용할 수 있음.

In [28]:
# 02. 도구에서 상태 업데이트하기.
# 이제 사람의 판단을 반영해 상태를 갱신할 수 잇도록 human_assistance 도구를 확장함.
# 이 도구 내부에서 interrupt() 를 호출해 실행을 일시 중지하고, 사용자의 응답에 따라 상태를 직접 업데이트함.

# 먼저 이 도구에 전달되는 파라미터의 의미를 간략히 살펴보겠음.
#   - @tool: 이 함수가 챗봇에서 호출 가능한 '도구'임을 나타냄.
#   - name, release_date : 사용자의 검토가 필요한 대상 정보임.
#   - tool_call_id : 해당 도구 호출로 고유 ID로, ToolMessage 생성 시 필수임.
#       . InjectedToolCallId 를 사용하면 이 값이 모델에게 노출 되지 않도록 숨길 수 있음.
from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.types import Command, interrupt

@tool
# 상태 업데이트를 위한 ToolMessage 를 생성하는 경우, 일반적으로 해당 도구 호출(tool call)에 대한 ID가 필요함.
# 이때 랭체인의 InjectedToolCallId 를 사용하면 해당 인자가 도구의 스키마(Schema)에 모델에게 노출되지 않도록 처리할 수 잇음.
def human_assistance(
    name: str, release_date: str, tool_call_id: Annotated[str, InjectedToolCallId]
):
    """사람의 확인이 필요한 정보를 검토 받는 도구입니다."""

    # 사람에게 정보가 맞는지 물어봄.
    human_response = interrupt(
        {
            "question": "이 정보가 맞나요?",
            "name": name,
            "release_date": release_date
        }
    )

    # 사람이 "Yes"로 응답한 경우 상태를 그대로 사용.
    if human_response.get("correct", "").lower().startswith("y"):
        verified_name = name
        verified_date = release_date
        response = "정보가 정확하다고 확인됨"

    # 수정이 필요한 경우 사람이 제공한 값을 사용.
    else:
        verified_name = human_response.get("name", name)
        verified_date = human_response.get("release_date", release_date)
        response = f"사람이 수정한 정보: {human_response}"

    # 상태에 name, release_date 를 저장하고, 메시지도 함께 반환.
    # ToolMessage
    # LLM이 호출한 도구(Tool)의 실행 결과를 다시 LLM에게 전달할때 사용하는 메시지 객체임.
    # 파이썬 함수나 API 를 실행한 후, 그 결과값을 다시 LLM을 보낼때 사용하는 규격임.
    # 속성
    #   . content : 도구가 실제 수행하고 반환한 결과 데이터(대개 문자열)
    #   . tool_call_id: 어떤 도구 호출 요청에 대한 답변인지 매칭해주는 고유 ID(LLM이 처음 부여해준 ID를 그대로 사용.)
    state_update = {
        "name": verified_name,
        "release_date": verified_date,
        "messages": [ToolMessage(response, tool_call_id = tool_call_id)]
    }

    # 상태 갱신을 위해 Command 객체를 반환.
    # 그래프의 흐름을 코드 레벨에서 동적으로 제어(Control)하기 위해 사용하는 매우 강력한 도구임.
    # 기존 Langgraph 에서는 다음 노드로 이동할 때 return "next_node"같은 문자열을 반환하거나 조건부 간선(Conditional Edge)를 미리 정해두어야 함.
    # 하지만 Command 를 사용하면 노드안에서 직접 "상태를 업데이트 하면서" 동시에 다음 노드를 지정할 수 있음.
    # 주로 다음과 같은 인자들을 받아 가면 사용됨.
    #   .update : 현재 그래프의 상태(State)를 변경할 데이터를 담음.
    #   .goto : 다음 실행할 노드의 이름을 지정함. (조건부 간선을 하드코딩하지 않고, 노드 내부로직에 따라 동적으로 분기할 때 유용함.)
    return Command(update = state_update)    

# 위 코드는 사람의 응답에 따라 상태 값을 수정하는 과정을 보여 줌.
#   ."correct"필드가 "y"로 시작하면 기존 정보를 그대로 사용하고,
#   . 그렇지 않은 경우라면 사용자가 입력한 수정값을 반영함.

# 이처럼 도구 내부에서 명시적으로 상태를 갱신하면 챗봇이 정보를 수집하고 검토받아 검증 워크플로우를 손쉽게 구현할 수 있음.

In [29]:
# 03. 챗봇 그래프 구성하기.
# 이번 단계에서는 챗봇 동작의 전체 흐름을 그래프 형태로 정의함.
# 앞서 정의한 챗봇 함수와 도구(human_assistance, TavilySearch)를 활용해 그래프를 구성하고, 상태를 자동 저장할 수 있도록 체크포인터도 연동함.
search_tool = TavilySearch(max_result = 2)
tools = [search_tool, human_assistance]
llm_with_tools = llm.bind_tools(tools)

# assert 문은 특정 조건이 참(True)인지 확인하고, 거짓(False)일 경우 AssertionError 를 발생 시켜 프로그램을 중단하는 디버깅용 명령어임.
# 개발 과정에서 "이 시점에는 반드시 이 조건이 성립해야 한다."는 가정을 검증할 때 사용함.
# 참고로 assert는 함수가 아니라 키워드(구문)이므로 괄호(())를 사용하지 않고 한칸 띄워서 작성함.
# 기본 문법
# 1. 메시지 생략 확인
#   assert 조건
# 2. 에러 메시지 추가 형식(권장)
#   assert 조건, "에러 발생 시 출력할 메시지"
# 조건이 True 일 때: 아무런 일도 일어나지 않고 다음 코드가 정상 실행됨.
# 조건이 False 일때 : 프로그램이 즉시 중단되며 지정한 메시지와 함께 AssertionError 가 발생함.
def chatbot(state:State):
    message = llm_with_tools.invoke(state["messages"])
    assert(len(message.tool_calls) <= 1) 
    return {"messages":[message]}  

graph_builder = StateGraph(State)

graph_builder.add_node("chatbot", chatbot)
tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

memory = MemorySaver()

graph = graph_builder.compile(checkpointer=memory)

In [ ]:
# 04. 질문 입력으로 챗봇 시작하기.
# 먼저 챗봇에게 다음과 같은 질문을 전달해 랭그래프의 출시일을 검색하고, 그 결과를 사람에게 확인받도록 요청함.
user_input = (
    "랭그래프가 언제 출시되었는지 찾아줄래요?"
    "결과를 찾으면 human_assistance 도구를 사용해서 사람에게 확인해 줘"
)

config = {"configurable": {"thread_id": "1"}}

events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config,
    stream_mode= "values"
)

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

#snapshot = graph.get_state(config)
#print(snapshot.next)

# 현재 그래프는 tools 노드에서 중단된 상태로, 사람이 응답할 때까지 기다라는 상태임.
# 이제 사람이 직접 응답을 제공해 흐름을 이어가 보겠음. 다음과 같이 Command 객체를 통해 수집된 정보를 전달하면 됨.
# Command 는 resume 속성을 사용하여 interrupt()의 답변으로 데이터를 전달함.
human_command = Command(
    resume = {
        "name": "랭그래프",
        "release_date": "2024년 1월 17일"
    },
)

events = graph.stream(human_command, config, stream_mode="values")

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

# 챗봇은 사람의 응답을 받아 상태를 갱신하고, 사용자에게 정확한 출시일을 안내함. 상태에 정보가 제대로 반영 됐는지 확인하려면 다음과 같이 get_state()를 사용해볼 수 있음.
snapshot = graph.get_state(config)
{k: v for k, v in snapshot.values.items() if k in ("name", "release_date")}

# [결과] : {'name': '랭그래프', 'release_date': '2024년 1월 17일'}

# 이렇게 저장된 값은 이후 노드에서 가공, 저장 또는 다시 검토하는데 활용할 수있음.

# 앞에서 살펴본 것처럼, 랭그래프는 상태를 자동으로 관리하면서도 필요한 경우 수동으로 값을 수정할 수있는 유연함을 제공함.
# 특히 사람이 개입하는 흐름 중에는 상태의 특정 키를 직접 덮어쓸수 있음.
# 예를 들어 이전에 지정한 name 값을 사람이 직접 수정하고자 한다면 다음과 같이 update_state()를 호출 하면 됨.
graph.update_state(config, {"name":"LangGraph (library)"})

# [결과] : {'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': 'if...'}}

# 이코드는 현재 설정된 config 에 따라 상태 중 name 키의 값을 "LangGraph (library)"로 직접 덮어씀
# 상태가 정상적으로 변경되었는지 확인하려면 get_state() 를 통해 현재 상태를 조회할 수 있음.
snapshot = graph.get_state(config)
{k: v for k, v in snapshot.values.items() if k in ("name", "release_date")}

# [결과] : {'name': 'LangGraph (library)', 'release_date': '2024년 1월 17일'}

In [ ]:
# 상태 이력 체크하기.
# 랭그래프에서는 get_state_history(config)을 통해 특정 스레드에 대한 전체 상태 이력(checkpoint 기록)을 가져올 수 있음.
# 이 정보는 그래프 실행 도중에 각 상태를 되짚어 보고 분석하거나, 특정 시점으로 되감기해서 다시 실행하고 싶을 대 유용함.

# 그래프의 상태 기록 전체를 가져옴. (지금까지의 실행 히스토리)
history = graph.get_state_history(config)
to_replay = None
for state in history:
    # 각 상태에서 메시지 갯수와 다음 노드를 출력함.
    print("Messages:", len(state.values["messages"]), "Next": state.next)
    print("-"*80)

    # 메시지가 6개인 상태를 선택함(임의 기준으로 선택)
    if(len(state.values["messages"]) == 3):
        # 나중에 되감기 (replay)할 상태를 저장함.
        to_replay = state

# [결과]
# Messages: 6 Next: ()
# ----------------------------------------------------------------------------------------
# Messages: 6 Next: ()
# ----------------------------------------------------------------------------------------
# Messages: 5 Next: ('chatbot',)
# ----------------------------------------------------------------------------------------
# Messages: 4 Next: ('tools',)
# ----------------------------------------------------------------------------------------
# Messages: 3 Next: ('chatbot',)
# ----------------------------------------------------------------------------------------
# Messages: 2 Next: ('tools',)
# ----------------------------------------------------------------------------------------
# Messages: 1 Next: ('chatbot',)
# ----------------------------------------------------------------------------------------
# Messages: 0 Next: ('__start__')

# 위 코드에서는 len(state.values["messages"]) == 3 이 메시지가 3개 까지 쌓인 시점을 되감기 기준으로 삼겠다는 의미임.
# 즉 메시지 3개가 쌓인 시점은 그래프가 처음 시작된 __start__노드 부터 "Message: 3"상태까지 임.

# 특정 시점에서 실행하기.
# 앞서 graph.get_state_history(config)을 통해 특정 시점의 상태를 to_replay 변수에 저장해 뒀음.
# 이 상태는 메시지가 3개까지 쌓인 시점이며, 곧 chatbot 노드가 실행된 이후 임.

# 이제 이 to_replay 상태에서 그래프 실행을 다시 시작해 보겠음.
print(to_replay.next)
print(to_replay.config)

# [결과]
# ('chatbot')
# {'configurable': {'thread_id': '1', 'checkpoint_ns':'', 'checkpoint_id': '1f0...'}}

# 여기서 to_replay.next 값이 ('chatbot',)이라는 것은 그래프가 재개될 경우 다음으로 실행될 노드가 chatbot 임을 의미함. 즉, 해당 체크포인트 시점 이후에 실행 흐름을 chatbot 노드 부터 다시 시작함.

# 이 시점을 기준으로 다시 실행.
# 체크포인트에는 to_replay.config에 타임 스탬프를 포함한 설정 정보가 저장돼 있으며, checkpoint_id 를 제공하면 랭그래프의 체크포인트가 해당 시점의 상태를 로드함.

# 저장된 시점으로 되감아 실행함.
for event in graph.stream(None, to_replay.config, stream_value="values"):
    if "messages" in event:
            event["messages"][-1].pretty_print()

# 실행 결과를 보면 도구 호출 이후의 흐름이 다시 재현됨. 이처럼 랭그래프는 과거의 특정 시점으로 되돌아가 실행을 제한하거나, 다른 분기의 탐색을 하는것을 매우 유연하게 지원함.